# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moezulhaq24/FlyRank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Signal checks

Before writing the baseline rule, I checked the two signals that the rule depends on.

**Signal 1 — Staleness**

I define a stale page as one that has not been updated for at least 90 days. I apply a minimum visibility floor of 500 impressions so that the comparison is not dominated by pages with almost no search exposure.

**Verdict: CONFIRMED**

In this slice, visible pages that had not been updated for 90+ days showed a directionally higher observed decline rate than visible pages updated within the last 90 days. The difference is not evidence that staleness causes decline, but it is enough to keep staleness as a simple baseline signal.

**Signal 2 — CTR relative to search position**

For pages with at least 500 impressions and an average position between 1 and 20, I compare pages with CTR below 0.5% against pages with CTR of at least 0.5%.

**Verdict: CONFIRMED**

Low-CTR pages in this visible top-20-position slice showed a higher observed decline rate. This supports using low CTR as an opportunity signal when the page already has meaningful visibility.

### My baseline rule

A page is prioritized for refresh review when:

- it has not been updated for at least 90 days,
- it received at least 500 search impressions in the trailing 90-day window,
- its average search position is between 1 and 20,
- and its CTR is below 0.5%.

Among pages that meet these conditions, pages receive a higher action score when they have more search visibility and a larger CTR gap below 0.5%.

The rule is intentionally simple and is designed as a decision-support baseline that the Week-5 machine-learning model must beat.

**Reason code:** `STALE_VISIBLE_LOW_CTR`

**Action label:** `REVIEW_FOR_REFRESH`

Pages that do not trigger the rule receive `NO_BASELINE_TRIGGER` and the action `MONITOR`.

In [5]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/content_refresh_anonymized.csv")

In [3]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Evaluation label
# IMPORTANT:
# This is used ONLY to audit signals and evaluate the rule.
# It is NOT used to calculate the baseline action score.
# ---------------------------------------------------------

is_declining_label = (
    df["trend_direction"].astype(str).str.lower().eq("down")
).astype(int)

audit_df = df.copy()
audit_df["is_declining_label"] = is_declining_label


# =========================================================
# SIGNAL 1 — STALENESS
# Only use pages with meaningful visibility.
# =========================================================

signal1 = audit_df[audit_df["impressions_90d"] >= 500].copy()

signal1["staleness_bucket"] = np.where(
    signal1["days_since_last_update"] >= 90,
    "90+ days",
    "<90 days"
)

staleness_table = (
    signal1
    .groupby("staleness_bucket")
    .agg(
        n=("content_id", "size"),
        decline_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

staleness_table["decline_rate"] = (
    staleness_table["decline_rate"] * 100
).round(1)

print("SIGNAL 1 — STALENESS")
print("Verdict: CONFIRMED")
display(staleness_table)


# =========================================================
# SIGNAL 2 — CTR VS POSITION
# Meaningful visibility + position 1–20
# =========================================================

signal2 = audit_df[
    (audit_df["impressions_90d"] >= 500)
    & (audit_df["avg_position"] > 0)
    & (audit_df["avg_position"] <= 20)
].copy()

signal2["ctr_bucket"] = np.where(
    signal2["ctr"] < 0.5,
    "CTR < 0.5%",
    "CTR >= 0.5%"
)

ctr_position_table = (
    signal2
    .groupby("ctr_bucket")
    .agg(
        n=("content_id", "size"),
        decline_rate=("is_declining_label", "mean"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median"),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

ctr_position_table["decline_rate"] = (
    ctr_position_table["decline_rate"] * 100
).round(1)

print("\nSIGNAL 2 — CTR VS POSITION")
print("Verdict: CONFIRMED")
display(ctr_position_table)

SIGNAL 1 — STALENESS
Verdict: CONFIRMED


,staleness_bucket,n,decline_rate,median_impressions,median_ctr
0,90+ days,6575,61.6,3435.0,0.15
1,<90 days,10151,58.2,2688.0,0.18



SIGNAL 2 — CTR VS POSITION
Verdict: CONFIRMED


,ctr_bucket,n,decline_rate,median_ctr,median_position,median_impressions
0,CTR < 0.5%,9759,62.7,0.17,8.4,3017.0
1,CTR >= 0.5%,2264,47.5,0.73,7.2,4729.5


## 2. Build the ranked queue (writes the CSV)

I convert the rule into a transparent action score rather than fitting any weights from the target.

A page must first pass four gates: at least 90 days since the last update, at least 500 impressions, a valid average position between 1 and 20, and CTR below 0.5%.

For pages that pass the gates, the score increases with search visibility and with the size of the CTR gap below 0.5%. I use log-scaled impressions only to reduce the effect of extremely large traffic values.

The score is normalized to a 0–100 range.

No label-derived fields, future outcome fields, client identifiers, or existing product flags are used to calculate the score.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================================================
# BUILD THE BASELINE SCORE
# =========================================================

work_df = df.copy()

# These are the ONLY inputs used by the scoring rule.
score_inputs = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

# Rule gates
stale = work_df["days_since_last_update"] >= 90

visible = work_df["impressions_90d"] >= 500

position_ok = (
    (work_df["avg_position"] > 0)
    & (work_df["avg_position"] <= 20)
)

# CTR values are percentages:
# 0.5 means 0.5%, not 50%.
ctr_gap = (
    1 - (work_df["ctr"] / 0.5).clip(lower=0, upper=1)
)

# Transparent score:
# If any gate fails, raw score becomes zero.
raw_score = (
    stale.astype(int)
    * visible.astype(int)
    * position_ok.astype(int)
    * ctr_gap
    * np.log1p(work_df["impressions_90d"])
)

# Normalize to 0–100
if raw_score.max() > 0:
    work_df["action_score"] = 100 * raw_score / raw_score.max()
else:
    work_df["action_score"] = 0.0


# One reason-code column; never multiple hidden reasons.
work_df["reason_code"] = np.where(
    work_df["action_score"] > 0,
    "STALE_VISIBLE_LOW_CTR",
    "NO_BASELINE_TRIGGER"
)

work_df["action_label"] = np.where(
    work_df["action_score"] > 0,
    "REVIEW_FOR_REFRESH",
    "MONITOR"
)


# Add label ONLY after scoring, for evaluation.
# It never entered action_score.
work_df["is_declining_label"] = is_declining_label


# Rank highest score first.
ranked = (
    work_df
    .sort_values(
        ["action_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked["baseline_rank"] = np.arange(1, len(ranked) + 1)


# =========================================================
# EVALUATE THE SIMPLE BASELINE
# =========================================================

base_rate = ranked["is_declining_label"].mean()

precision_at_20 = (
    ranked.head(20)["is_declining_label"].mean()
)

precision_at_50 = (
    ranked.head(50)["is_declining_label"].mean()
)

triggered_rows = int((ranked["action_score"] > 0).sum())


print("Total rows:", len(ranked))
print("Rows triggering rule:", triggered_rows)
print(f"Base declining rate: {base_rate:.3f}")
print(f"Precision@20: {precision_at_20:.3f}")
print(f"Precision@50: {precision_at_50:.3f}")


# =========================================================
# WRITE THE REQUIRED CSV
# Label / future outcome columns are NOT included.
# =========================================================

output_columns = [
    "baseline_rank",
    "content_id",
    "client_id",
    "action_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "sessions_90d",
    "engagement_rate"
]

queue_output = ranked[output_columns].copy()

OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / "baseline_action_score.csv"

queue_output.to_csv(CSV_PATH, index=False)

print("\nQueue written to:")
print(CSV_PATH)

display(queue_output.head(20))

Total rows: 30000
Rows triggering rule: 3534
Base declining rate: 0.542
Precision@20: 0.550
Precision@50: 0.580

Queue written to:
baseline_action_score.csv


,baseline_rank,content_id,client_id,action_score,reason_code,action_label,days_since_last_update,impressions_90d,avg_position,ctr,sessions_90d,engagement_rate
0,1,content_c8e9d6ab9013,client_19581e27de,100.000000,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,208678,9.7,0.00,6,0.00
1,2,content_4a6607efcb46,client_6208ef0f77,94.093714,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,128068,2.2,0.01,87,2.30
2,3,content_36ff89c8214e,client_19581e27de,92.546091,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,295097,7.3,0.05,238,1.68
3,4,content_c1fe78bc4e37,client_19581e27de,90.603785,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,134055,7.5,0.03,170,1.18
4,5,content_b115f7c74779,client_19581e27de,89.972495,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,123469,8.0,0.03,43,2.33
5,6,content_d0cc5baa4995,client_19581e27de,86.984608,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,83651,6.6,0.03,69,1.45
6,7,content_91652435f57a,client_19581e27de,86.073234,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,159590,7.8,0.06,103,0.00
7,8,content_d07ea098353c,client_19581e27de,84.853264,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,63366,9.4,0.03,30,3.33
8,9,content_9c8299b55f3c,client_624b60c58c,83.736297,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,54783,8.5,0.03,37,0.00
9,10,content_42634cb0c5a3,client_6208ef0f77,83.651668,STALE_VISIBLE_LOW_CTR,REVIEW_FOR_REFRESH,104,43175,18.8,0.02,97,1.03


In [ ]:
metrics = {
    "rows": int(len(ranked)),
    "triggered_rows": triggered_rows,
    "base_declining_rate": round(float(base_rate), 4),
    "precision_at_20": round(float(precision_at_20), 4),
    "precision_at_50": round(float(precision_at_50), 4),
    "rule": {
        "days_since_last_update_min": 90,
        "impressions_90d_min": 500,
        "avg_position_min": 1,
        "avg_position_max": 20,
        "ctr_max_percent": 0.5
    }
}

METRICS_PATH = OUTPUT_DIR / "baseline_metrics.json"

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics written to:")
print(METRICS_PATH)

## 3. Top-20 review

I reviewed the top 20 recommendations using only information that would be available when the action decision is made.

The review is intentionally skeptical. A high baseline score means that the page satisfies the rule; it does not prove that refreshing the page will improve traffic.

Confidence is higher when a page has strong search visibility, a good existing search position, and an unusually low CTR. Confidence is lower when the page ranks closer to positions 11–20, has sparse engagement data, or already shows relatively strong on-page engagement.

For every recommendation I also record what could make the rule wrong. Examples include query intent, SERP features, seasonality, limited GA4 observations, or low CTR being normal for the page's actual ranking position.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================================================
# TOP-20 HUMAN REVIEW
# =========================================================

top20 = ranked.head(20).copy()

# A relative engagement threshold for skeptical review only.
# This does NOT affect the action score.
engagement_p90 = df["engagement_rate"].quantile(0.90)


def confidence_note(row):
    """
    Human-readable confidence description.
    Uses only decision-time signals.
    """

    if (
        row["avg_position"] <= 10
        and row["ctr"] < 0.10
        and row["impressions_90d"] >= 3000
    ):
        return (
            "HIGH — strong search visibility with extremely low CTR."
        )

    if row["avg_position"] <= 10:
        return (
            "MEDIUM-HIGH — page-one visibility makes the low CTR worth reviewing."
        )

    return (
        "MEDIUM — the page has visibility, but its lower position "
        "may explain part of the low CTR."
    )


def wrong_note(row):
    """
    One skeptical explanation for why the baseline recommendation
    may not be the correct action.
    """

    reasons = []

    if row["avg_position"] > 10:
        reasons.append(
            "CTR may be normal for a position outside page one"
        )

    if (
        row["engagement_rate"] >= engagement_p90
        and row["sessions_90d"] >= 20
    ):
        reasons.append(
            "on-page engagement is relatively strong, so a full refresh may be unnecessary"
        )

    if row["sessions_90d"] < 30:
        reasons.append(
            "GA4 activity is sparse, so engagement evidence is weak"
        )

    if not reasons:
        reasons.append(
            "query intent, SERP features, or seasonality could explain the low CTR"
        )

    return "; ".join(reasons) + "."


top20["why_it_is_here"] = top20.apply(
    lambda row:
        f"{int(row['days_since_last_update'])} days since update; "
        f"{int(row['impressions_90d']):,} impressions; "
        f"position {row['avg_position']:.1f}; "
        f"CTR {row['ctr']:.2f}%.",
    axis=1
)

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_note,
    axis=1
)


review_columns = [
    "baseline_rank",
    "content_id",
    "action_label",
    "reason_code",
    "why_it_is_here",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns]

display(top20_review)

,baseline_rank,content_id,action_label,reason_code,why_it_is_here,confidence_note,what_would_make_it_wrong
0,1,content_c8e9d6ab9013,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 208,678 impressions; po...",HIGH — strong search visibility with extremely...,"GA4 activity is sparse, so engagement evidence..."
1,2,content_4a6607efcb46,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 128,068 impressions; po...",HIGH — strong search visibility with extremely...,"query intent, SERP features, or seasonality co..."
2,3,content_36ff89c8214e,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 295,097 impressions; po...",HIGH — strong search visibility with extremely...,"query intent, SERP features, or seasonality co..."
3,4,content_c1fe78bc4e37,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 134,055 impressions; po...",HIGH — strong search visibility with extremely...,"query intent, SERP features, or seasonality co..."
4,5,content_b115f7c74779,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 123,469 impressions; po...",HIGH — strong search visibility with extremely...,"query intent, SERP features, or seasonality co..."
5,6,content_d0cc5baa4995,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 83,651 impressions; pos...",HIGH — strong search visibility with extremely...,"query intent, SERP features, or seasonality co..."
6,7,content_91652435f57a,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 159,590 impressions; po...",HIGH — strong search visibility with extremely...,"query intent, SERP features, or seasonality co..."
7,8,content_d07ea098353c,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 63,366 impressions; pos...",HIGH — strong search visibility with extremely...,"query intent, SERP features, or seasonality co..."
8,9,content_9c8299b55f3c,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 54,783 impressions; pos...",HIGH — strong search visibility with extremely...,"query intent, SERP features, or seasonality co..."
9,10,content_42634cb0c5a3,REVIEW_FOR_REFRESH,STALE_VISIBLE_LOW_CTR,"104 days since update; 43,175 impressions; pos...","MEDIUM — the page has visibility, but its lowe...",CTR may be normal for a position outside page ...


## 4. Weak picks + leakage check

### Weak picks

The baseline is intentionally simple, so I do not expect every top-ranked recommendation to be correct.

I treat recommendations as weaker when the page ranks outside the first search-results page because low CTR may partly be explained by position rather than by stale content.

I also treat a recommendation as weaker when its on-page engagement is relatively strong. In that case, the content itself may still satisfy users and a CTR/snippet review could be more appropriate than a full content refresh.

These cases show why the ranked queue should support human review rather than automatically trigger content changes.

### Leakage check

The action score uses only:

- `days_since_last_update`
- `impressions_90d`
- `avg_position`
- `ctr`

The target is derived from `trend_direction`, so `trend_direction`, `trend_pct`, and `is_declining_label` are excluded from the score.

I also exclude recent-window outcome fields such as `impressions_last_30d`, `clicks_last_30d`, and other last/previous-window fields from the baseline rule.

Client and content identifiers are retained only as context and are never scoring features.

No existing product flag is used to calculate the baseline score.

In [12]:
# =========================================================
# WEAK PICKS
# =========================================================

weak_picks = top20[
    (top20["avg_position"] > 10)
    |
    (
        (top20["engagement_rate"] >= engagement_p90)
        & (top20["sessions_90d"] >= 20)
    )
].copy()

print("Weak picks found in Top 20:", len(weak_picks))

display(
    weak_picks[
        [
            "baseline_rank",
            "content_id",
            "action_score",
            "impressions_90d",
            "avg_position",
            "ctr",
            "sessions_90d",
            "engagement_rate",
            "what_would_make_it_wrong"
        ]
    ]
)


# =========================================================
# LEAKAGE CHECK
# =========================================================

score_inputs = {
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
}

label_derived_fields = {
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

recent_window_fields = {
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
}

identifier_fields = {
    "content_id",
    "client_id"
}

# Search dataset for any columns that look like existing flags.
product_flag_columns = [
    column
    for column in df.columns
    if "flag" in column.lower()
]


assert score_inputs.isdisjoint(label_derived_fields), \
    "ERROR: label-derived field leaked into score."

assert score_inputs.isdisjoint(recent_window_fields), \
    "ERROR: recent outcome window leaked into score."

assert score_inputs.isdisjoint(identifier_fields), \
    "ERROR: identifier leaked into score."


print("\nLEAKAGE CHECK")
print("Score inputs:", sorted(score_inputs))
print("Label-derived fields used in score: NONE")
print("Recent comparison-window fields used in score: NONE")
print("Identifiers used in score: NONE")
print("Product-flag-like columns found in raw data:", product_flag_columns)

print("\n✅ Leakage check passed.")

Weak picks found in Top 20: 3


,baseline_rank,content_id,action_score,impressions_90d,avg_position,ctr,sessions_90d,engagement_rate,what_would_make_it_wrong
9,10,content_42634cb0c5a3,83.651668,43175,18.8,0.02,97,1.03,CTR may be normal for a position outside page ...
17,18,content_8223440cd40c,80.309185,35051,17.9,0.03,76,2.63,CTR may be normal for a position outside page ...
19,20,content_a38dd531fd8f,80.256434,22716,6.5,0.01,25,12.00,"on-page engagement is relatively strong, so a ..."



LEAKAGE CHECK
Score inputs: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d']
Label-derived fields used in score: NONE
Recent comparison-window fields used in score: NONE
Identifiers used in score: NONE
Product-flag-like columns found in raw data: []

✅ Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.